# A2: reference-frequency sampling and complete-chain checkpoints

Same scientific A2 model, the 15 retrained checkpoints, and the same Bernoulli localisation response. This is an execution revision, not a new amplitude-recovery model. The Amazon-only experiment requires a separate statistical specification.

Start in **PILOT**. Existing compatible FULL/PILOT measurements can be reused, including measurements from a run whose MCMC failed. Old posteriors are never imported into this run. Use `SOURCE_RUN_OVERRIDE` if the automatic source choice is not the desired population. Environment/provenance checks remain enforced.

Changes: exact reference-frequency coordinates at 64 Hz; dense NUTS mass adaptation; one independently seeded chain at a time, saved and reloaded with completeness validation; plain-text progress without widgets; additional sparse-data recovery and frequency-level PPC gates. A completed chain can be reused after interruption, but an unfinished chain restarts its own warmup. Samples are never selected by convergence or divergence count.

Default retained draws: PILOT 2,000 per chain, FULL 3,000 per chain, 4 chains, warmup 2,000. The draws setting does not change the scientific pilot/full compatibility fingerprint. A change to a previously existing run's sampling settings requires a new run namespace.

**Validation limit:** the delivered notebook has no empirical results. Read the accompanying validation report for software and bounded-run checks. Passing a small/synthetic check does not establish FULL convergence.


## 1. Setup
### 1.1 Locate the repository
Analysis code is embedded here; the frozen repository provides signal generators and the checkpoint loader.


In [ ]:
import importlib.util, os, subprocess, sys
from pathlib import Path
REPO_URL = "https://github.com/FedericoSabbadini/patchAliasing.git"
REPO_REVISION = "9d478e9a5aada82d138a47262ab5c2dc592911bc"
MARKER = Path("chronos/bayesian/probe_lib.py")
def on_colab():
    try:
        return importlib.util.find_spec("google.colab") is not None
    except ModuleNotFoundError:
        return False
IS_COLAB = on_colab()
def find_repo():
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / MARKER).is_file():
            return candidate
    target = Path("/content/patchAliasing") if IS_COLAB else here / "patchAliasing"
    if not (target / MARKER).is_file():
        subprocess.check_call(["git", "clone", "--no-checkout", REPO_URL, str(target)])
        subprocess.check_call(["git", "-C", str(target), "checkout", "--detach", REPO_REVISION])
    return target
REPO = find_repo()
BAYES_DIR = REPO / "chronos/bayesian"
sys.path.insert(0, str(BAYES_DIR)) if str(BAYES_DIR) not in sys.path else None
print("Repository:", REPO)


### 1.2 Environment
Colab installs the repository lock and verifies fresh imports after a deliberate restart. Local execution uses the active project environment. The posterior code supports both InferenceData and DataTree.


In [ ]:
import json
import shutil
import sysconfig
import tempfile
import time

RESTART_STATE_PATH = Path(tempfile.gettempdir()) / "patchaliasing_A2_env_state.json"
MAX_RESTARTS = 2


def uv_executable() -> str:
    found = shutil.which("uv")
    if found:
        return found
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "uv"])
    candidates = [
        Path(sysconfig.get_path("scripts")) / ("uv.exe" if os.name == "nt" else "uv"),
        Path(sys.executable).parent / ("uv.exe" if os.name == "nt" else "uv"),
    ]
    for candidate in candidates:
        if candidate.is_file():
            return str(candidate)
    raise FileNotFoundError("uv was installed but its executable was not found")


def load_restart_state() -> dict:
    if RESTART_STATE_PATH.is_file():
        try:
            return json.loads(RESTART_STATE_PATH.read_text(encoding="utf-8"))
        except json.JSONDecodeError:
            pass
    return {"restarts": 0}


def save_restart_state(state: dict) -> None:
    RESTART_STATE_PATH.write_text(json.dumps(state), encoding="utf-8")


def clean_imports_are_healthy() -> bool:
    probe = subprocess.run(
        [
            sys.executable,
            "-c",
            "import numpy, scipy, pandas, pyarrow, arviz, pymc, nutpie, h5netcdf",
        ],
        capture_output=True,
        text=True,
    )
    if probe.returncode != 0:
        print(probe.stderr[-2500:])
    return probe.returncode == 0


def restart_colab(reason: str) -> None:
    print("=" * 78)
    print(f"RESTARTING THE COLAB RUNTIME: {reason}")
    print("This is deliberate. After reconnection choose Runtime > Run all again.")
    print("=" * 78)
    sys.stdout.flush()
    time.sleep(2)
    os.kill(os.getpid(), 9)


if not IS_COLAB:
    print("Local runtime: dependency installation skipped; using the active environment.")
else:
    state = load_restart_state()
    UV = uv_executable()

    if state["restarts"] == 0:
        with tempfile.TemporaryDirectory() as temporary:
            requirements = Path(temporary) / "requirements.locked.txt"
            subprocess.check_call(
                [
                    UV,
                    "export",
                    "--frozen",
                    "--no-dev",
                    "--no-emit-project",
                    "--no-hashes",
                    "--output-file",
                    str(requirements),
                ],
                cwd=REPO,
            )
            subprocess.check_call(
                [UV, "pip", "install", "--python", sys.executable, "--requirement", str(requirements)]
            )

        # Required for saving the small posterior checkpoint on Python 3.12+.
        subprocess.check_call(
            [UV, "pip", "install", "--python", sys.executable, "h5netcdf==1.8.1", "h5py==3.16.0"]
        )

        # Colab's preinstalled vision wheels can be ABI-incompatible with the locked torch.
        # They are not used by this analysis.
        subprocess.run(
            [sys.executable, "-m", "pip", "uninstall", "-y", "torchvision", "torchaudio"],
            check=False,
            capture_output=True,
        )
        save_restart_state({"restarts": 1})
        restart_colab("locked environment installed")

    elif state["restarts"] == 1:
        if clean_imports_are_healthy():
            save_restart_state({"restarts": "verified"})
            print("Locked environment verified in the restarted runtime.")
        else:
            with tempfile.TemporaryDirectory() as temporary:
                requirements = Path(temporary) / "requirements.locked.txt"
                subprocess.check_call([UV, "export", "--frozen", "--no-dev", "--no-emit-project",
                                       "--no-hashes", "--output-file", str(requirements)], cwd=REPO)
                subprocess.check_call([UV, "pip", "install", "--python", sys.executable,
                                       "--reinstall-package", "numpy", "--reinstall-package", "scipy",
                                       "--requirement", str(requirements)])
            save_restart_state({"restarts": 2})
            restart_colab("NumPy/SciPy clean reinstall")
    else:
        if not clean_imports_are_healthy():
            raise RuntimeError(
                "The scientific imports are still broken after two restarts. Choose Runtime > "
                "Disconnect and delete runtime, reconnect to a fresh VM, and run all again."
            )
        save_restart_state({"restarts": "verified"})
        print("Locked environment verified.")


### 1.3 Settings
AUTO looks for A1 arm tables and their manifests on Drive. `A2_SOURCE_RUN` may name another compatible run root. Existing A1 and A2 outputs use separate namespaces.


In [ ]:
import gc, hashlib, inspect, json, platform, re, time
from pathlib import Path
from importlib import metadata as importlib_metadata
import numpy as np
import pandas as pd
import pymc as pm
import arviz as az
import xarray as xr
import matplotlib.pyplot as plt
from scipy.special import expit, ndtr, logit, gammaln
from IPython.display import display
import checkpointing as cp
import probe_lib as pl
import model_loader as ml

RUN_MODE = "PILOT"
RUN_ID = "model_A2_reference64_v1"
MODEL_VERSION = "A2-varying-lock-and-side-v1"
NOTEBOOK_VERSION = "A2-reference64-chain-checkpoints-v1"
SEED = 42
TONE_SNR, TOP_K, TOL_HZ, NFFT, N_PHASE = 1.25, 3, 1.0, 8192, 10
MODELS, GENERATORS = list(pl.DELIVERABLE3_MODELS), tuple(pl.GENERATORS)
PRIOR_SCALE, BASELINE_SCALE, NU = .5, 1.5, 4
PRIOR_SCALES = (.25, .5, 1.)
SUPPORT_LOG_OR, ROPE_LOG_OR, PROB_CUTOFF = float(np.log(.8)), float(np.log(1.1)), .95
RHAT_MAX, PILOT_ESS_MIN, FULL_ESS_MIN = 1.01, 400, 1000
PPC_MIN_COVERAGE, SENSITIVITY_MAX_SPREAD, RECOVERY_MIN_COVERAGE = .90, .10, .80
if RUN_MODE not in {"PILOT", "FULL"}:
    raise ValueError("RUN_MODE must be PILOT or FULL")
IS_FULL = RUN_MODE == "FULL"
N_BG = 100 if IS_FULL else 3
DRAWS, TUNE, CHAINS = (3000 if IS_FULL else 2000), 2000, 4
CORES = max(1, min(CHAINS, os.cpu_count() or 1))
TARGET_ACCEPT = .95
RECOVERY_DRAWS, RECOVERY_TUNE = 2000, 1500
PPC_DRAWS, EFFECT_DRAWS = 800, 2000
BATCH_SIZE, BG_PER_SHARD, SPECTRAL_BATCH = 64, 10, 256
NUTS_BACKEND = "pymc"
NUTS_INIT = "jitter+adapt_full"
REFERENCE_HZ = 64.0
PROGRESS_EVERY = 200
FINE_PPC_MIN_COVERAGE = .90
# Sequential chains: each completed chain is saved before the next starts.
# CPU sampling; CUDA remains available for Chronos collection.
COLLECTION_DEVICE = "auto"  # CUDA when available; Bayesian inference always uses CPU.
DATA_REUSE = "AUTO"  # OFF collects new arms and backgrounds in this run's namespace.
SOURCE_RUN_OVERRIDE = None  # Optionally point to an existing compatible A2 run.
if IS_COLAB:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
default_root = Path("/content/drive/MyDrive/patchAliasing") if IS_COLAB else BAYES_DIR/"_run"
DRIVE_ROOT = Path(os.environ.get("A2_DRIVE_ROOT", str(default_root)))
OUTPUT_ROOT = DRIVE_ROOT/("full" if IS_FULL else "pilots")/RUN_ID
PILOT_RESULT_PATH = DRIVE_ROOT/"pilots"/RUN_ID/"final_verdict.json"
DATA_ROOT, CHECKPOINT_ROOT, FIGURE_ROOT = (OUTPUT_ROOT/name for name in ("data","checkpoints","figures"))
MANIFEST_PATH = OUTPUT_ROOT/"analysis_manifest.json"
for directory in (OUTPUT_ROOT, DATA_ROOT, CHECKPOINT_ROOT, FIGURE_ROOT):
    directory.mkdir(parents=True,exist_ok=True)
pl.TONE_SNR = TONE_SNR
np.random.seed(SEED)
plt.style.use(next((s for s in ("arviz-whitegrid","seaborn-v0_8-whitegrid") if s in plt.style.available),"default"))
print("Mode:",RUN_MODE,"| MCMC:",NUTS_BACKEND,NUTS_INIT,"| one chain at a time")
print("PyMC / ArviZ / Python:",pm.__version__,az.__version__,platform.python_version())
print("Backgrounds per generator:",N_BG,"| draws/tune/chains:",DRAWS,TUNE,CHAINS)
print("Output:",OUTPUT_ROOT)


## 2. Pipeline definitions
These definitions are fingerprinted before collecting or sampling. Source-data compatibility checks concern measurement and checkpoint provenance, not equality of the A1/A2 statistical models.


In [ ]:
def frequency_design():
    rows = []
    for P, S in MODELS:
        for f_lock in pl.f_lock(P, S):
            delta = pl.control_offset(P, S, f_lock)
            if not np.isfinite(delta):
                continue
            phases = pl.phases_Sf(f_lock, N_PHASE)
            for phase_idx, phase in enumerate(phases):
                for role, f in (("lock", f_lock), ("lo", f_lock-delta), ("hi", f_lock+delta)):
                    rows.append(dict(model=pl.model_tag(P,S), P=P, S=S, overlap=(P-S)/P,
                                     f_lock=float(f_lock), delta=float(delta), phase_idx=phase_idx,
                                     phase=float(phase), role=role, f=float(f), is_lock=int(role=="lock")))
    return pd.DataFrame(rows)

def spectral_peaks(values):
    values = np.atleast_2d(np.asarray(values, dtype=float))
    if values.shape[1] != pl.PRED or not np.isfinite(values).all():
        raise ValueError("Expected finite forecast horizons only")
    pieces = []
    for start in range(0, len(values), SPECTRAL_BATCH):
        block = values[start:start+SPECTRAL_BATCH]
        peaks = pl.dominant_freqs(block, k=TOP_K, nfft=NFFT, band=pl.BAND)
        # A truly flat forecast has no spectral peak. Avoid argmax assigning the first band bin.
        peaks[np.ptp(block, axis=1) == 0] = np.nan
        pieces.append(peaks)
    return np.concatenate(pieces)

def hit_values(peaks, frequencies):
    return pl.localisation_hit(peaks, frequencies, tol=TOL_HZ)

def package_version(name):
    try:
        return importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        return None

def logic_fingerprint(functions):
    # Code-object content fingerprints the executing functions without relying on notebook paths.
    # Exclude filenames/line numbers, which change between otherwise identical Colab executions.
    import types
    def normalise(value):
        if isinstance(value, types.CodeType):
            return dict(code=value.co_code.hex(), names=value.co_names, variables=value.co_varnames,
                        constants=[normalise(x) for x in value.co_consts],
                        argcount=value.co_argcount, kwonly=value.co_kwonlyargcount,
                        freevars=value.co_freevars, cellvars=value.co_cellvars)
        if isinstance(value, (tuple, list)):
            return [normalise(x) for x in value]
        if isinstance(value, (set, frozenset)):
            # Set display order depends on PYTHONHASHSEED; a clean kernel must keep the same hash.
            return {"set":sorted((normalise(x) for x in value), key=cp.canonical_json)}
        if isinstance(value, (str, int, float, bool)) or value is None:
            return value
        return repr(value)
    return cp.fingerprint({f.__name__:normalise(f.__code__) for f in functions})

def read_manifest():
    current = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    if current.get("analysis_fingerprint") != ANALYSIS_FINGERPRINT:
        raise ValueError("Run manifest changed; do not combine different analyses")
    return current

def record_artifact(path):
    path = Path(path)
    manifest = read_manifest()
    manifest["artifacts"][path.relative_to(OUTPUT_ROOT).as_posix()] = {
        "sha256":cp.sha256_file(path), "bytes":path.stat().st_size}
    cp.atomic_json(MANIFEST_PATH, manifest)

def valid_artifact(path):
    path = Path(path)
    entry = read_manifest()["artifacts"].get(path.relative_to(OUTPUT_ROOT).as_posix())
    if path.is_file() != (entry is not None):
        raise ValueError(f"Untracked or missing artifact: {path}")
    if entry is None:
        return False
    if cp.sha256_file(path) != entry["sha256"]:
        raise ValueError(f"Artifact hash mismatch: {path}")
    return True

def save_table(path, frame):
    cp.atomic_parquet(path, frame)
    record_artifact(path)

def validate_arms(frame, design, n_bg=None):
    n_bg = N_BG if n_bg is None else n_bg
    keys = ["model", "generator", "bg_id", "f_lock", "phase_idx", "role"]
    if len(frame) != len(design)*len(GENERATORS)*n_bg or frame[keys].duplicated().any():
        raise ValueError("Incomplete/duplicate A-only arm design")
    if set(frame["generator"]) != set(GENERATORS) or set(frame["bg_id"]) != set(range(n_bg)):
        raise ValueError("Wrong background population")
    merged = frame.merge(design, on=["model", "f_lock", "phase_idx", "role"],
                         suffixes=("", "_expected"), how="left", validate="many_to_one")
    for name in ("P", "S", "overlap", "phase", "f", "delta", "is_lock"):
        if not np.allclose(merged[name], merged[name+"_expected"], rtol=0, atol=1e-9):
            raise ValueError(f"Arm metadata differs from the frozen design: {name}")
    sizes = frame.groupby(["generator", "bg_id"]).size()
    if len(sizes) != len(GENERATORS)*n_bg or not sizes.eq(len(design)).all():
        raise ValueError("Unequal or missing arm coverage per background")
    for name in ("h", "h_truth", "h_blind", "is_lock"):
        if not frame[name].isin([0,1]).all():
            raise ValueError(f"{name} must be binary")


### 2.1 Reuse, collection and matched background forecasts
Completed geometry/background shards are resumable. Reusing A1 copies both its validated arm rows and its exact stored backgrounds. The new baseline is a Chronos forecast of that same background context without an injected tone.


In [ ]:
def source_manifest(source):
    path = Path(source["root"])/"analysis_manifest.json"
    if cp.sha256_file(path) != source["manifest_sha256"]:
        raise ValueError("Source manifest changed during this run")
    return json.loads(path.read_text(encoding="utf-8"))

def source_artifact(source, relative):
    path = Path(source["root"])/relative
    entry = source_manifest(source).get("artifacts",{}).get(relative)
    if entry is None or not path.is_file() or cp.sha256_file(path)!=entry["sha256"]:
        raise ValueError(f"Source artifact missing or changed: {path}")
    return path

def resolve_source():
    if DATA_REUSE not in {"AUTO","OFF"}:
        raise ValueError("DATA_REUSE must be AUTO or OFF")
    if DATA_REUSE=="OFF":
        return None
    candidates = [Path(SOURCE_RUN_OVERRIDE)] if SOURCE_RUN_OVERRIDE else [
        DRIVE_ROOT/"full/model_A2_localisation_bernoulli_v2_3000_draws",
        DRIVE_ROOT/"full/model_A2_localisation_bernoulli_v2",
        DRIVE_ROOT/"full/model_A_localisation_bernoulli_v1",
        *([DRIVE_ROOT/"pilots"/RUN_ID] if IS_FULL else []),
        DRIVE_ROOT/"pilots/model_A2_localisation_bernoulli_v2",
        DRIVE_ROOT/"pilots/model_A2_localisation_bernoulli_v1",
        DRIVE_ROOT/"pilots/model_A_localisation_bernoulli_v1"]
    for root in candidates:
        path = root/"data/A_arms.parquet"
        if not path.is_file():
            continue
        if root.resolve()==OUTPUT_ROOT.resolve():
            raise ValueError("The source and destination run must differ")
        manifest_path = root/"analysis_manifest.json"
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        spec = manifest["analysis_spec"]
        if cp.fingerprint(spec)!=manifest.get("analysis_fingerprint"):
            raise ValueError("Source analysis fingerprint mismatch")
        science = spec["science"]
        if science.get("model") not in {"A-bernoulli-centred-zerosum-exact-counts-v1",MODEL_VERSION}:
            raise ValueError("Source is not a compatible A1/A2 localisation run")
        checks = dict(snr=TONE_SNR,top_k=TOP_K,tolerance_hz=TOL_HZ,nfft=NFFT,n_phase=N_PHASE,seed=SEED,scope="all_arms")
        for key,value in checks.items():
            if science.get(key)!=value:
                raise ValueError(f"Incompatible source measurement: {key}; select another source or DATA_REUSE='OFF'")
        if (cp.fingerprint(science.get("models"))!=cp.fingerprint(MODELS)
                or tuple(science.get("generators",[]))!=GENERATORS
                or science.get("checkpoints")!=CHECKPOINT_IDENTITIES):
            raise ValueError("Source geometry/generator/checkpoint population differs")
        for relative,digest in MEASUREMENT_HELPER_HASHES.items():
            if science.get("helper_hashes",{}).get(relative)!=digest:
                raise ValueError(f"Source measurement helper differs: {relative}")
        for package in ("numpy","torch","chronos-forecasting"):
            if spec.get("packages",{}).get(package)!=package_version(package):
                raise ValueError(f"Source environment differs for {package}; use its environment or DATA_REUSE='OFF'")
        source_n_bg = int(spec["n_bg"])
        if spec.get("run_mode") not in {"PILOT","FULL"} or source_n_bg<1:
            raise ValueError("Source is not a PILOT/FULL population")
        result = dict(root=str(root),manifest_sha256=cp.sha256_file(manifest_path),
                      analysis_fingerprint=manifest["analysis_fingerprint"],n_bg=min(N_BG,source_n_bg),
                      source_n_bg=source_n_bg,kind=science["model"])
        result["arms_sha256"] = cp.sha256_file(source_artifact(result,"data/A_arms.parquet"))
        # Stored backgrounds, not regenerated approximations, make the extra baseline paired.
        for generator in GENERATORS:
            for bg_id in range(result["n_bg"]):
                source_artifact(result,f"data/backgrounds/{generator}_{bg_id:03d}.npy")
        return result
    if SOURCE_RUN_OVERRIDE:
        raise FileNotFoundError(f"No data/A_arms.parquet under {SOURCE_RUN_OVERRIDE}")
    return None

def canonical_background(generator,bg_id):
    path = DATA_ROOT/"backgrounds"/f"{generator}_{bg_id:03d}.npy"
    if valid_artifact(path):
        values = np.load(path)
    else:
        if SOURCE is not None and bg_id<SOURCE["n_bg"]:
            values = np.load(source_artifact(SOURCE,f"data/backgrounds/{generator}_{bg_id:03d}.npy"))
        else:
            values = pl.background(generator,pl.CANON_LEN,10000+bg_id)
        cp.atomic_npy(path,values)
        record_artifact(path)
    if values.shape!=(pl.CANON_LEN,) or not np.isfinite(values).all():
        raise ValueError("Invalid background horizon")
    if abs(float(values.mean()))>1e-5 or abs(float(values.std())-1)>1e-5:
        raise ValueError("Expected a centred, unit-variance canonical background")
    return values

def collection_device():
    if COLLECTION_DEVICE in {"cpu","cuda"}:
        return COLLECTION_DEVICE
    if COLLECTION_DEVICE!="auto":
        raise ValueError("COLLECTION_DEVICE must be auto, cpu or cuda")
    import torch
    return "cuda" if torch.cuda.is_available() else "cpu"

def validate_calibration(frame):
    keys = ["model","generator","bg_id"]
    expected = {(pl.model_tag(P,S),g,b) for P,S in MODELS for g in GENERATORS for b in range(N_BG)}
    if frame[keys].duplicated().any() or set(map(tuple,frame[keys].to_numpy()))!=expected:
        raise ValueError("Incomplete/duplicate background forecasts")
    signals = frame[[f"forecast_{i}" for i in range(pl.PRED)]].to_numpy(float)
    peaks = spectral_peaks(signals)
    stored = frame[[f"background_f_hat_{i+1}" for i in range(TOP_K)]].to_numpy(float)
    np.testing.assert_allclose(stored,peaks,rtol=0,atol=1e-9,equal_nan=True)

def attach_calibration(arms,calibration):
    columns = [f"background_f_hat_{i+1}" for i in range(TOP_K)]
    clean = arms.drop(columns=[*columns,"h_background_forecast"],errors="ignore")
    frame = clean.merge(calibration[["model","generator","bg_id",*columns]],on=["model","generator","bg_id"],how="left",validate="many_to_one",indicator=True)
    if not frame._merge.eq("both").all():
        raise ValueError("Some arms lack a matched background-only forecast")
    frame = frame.drop(columns="_merge")
    frame["h_background_forecast"] = hit_values(frame[columns].to_numpy(float),frame.f.to_numpy(float))
    return frame

def collect_A2(design):
    merged_path,baseline_path = DATA_ROOT/"A_arms.parquet",DATA_ROOT/"background_forecasts.parquet"
    if valid_artifact(merged_path) and valid_artifact(baseline_path):
        arms,calibration = pd.read_parquet(merged_path),pd.read_parquet(baseline_path)
        validate_arms(arms,design)
        validate_calibration(calibration)
        expected = attach_calibration(arms,calibration)
        np.testing.assert_array_equal(expected.h_background_forecast,arms.h_background_forecast)
        return arms,calibration
    source_arms,source_baseline = None,None
    if SOURCE is not None:
        source_arms = pd.read_parquet(source_artifact(SOURCE,"data/A_arms.parquet"))
        source_arms = source_arms[source_arms.bg_id<SOURCE["n_bg"]].copy()
        validate_arms(source_arms,design,n_bg=SOURCE["n_bg"])
        if "data/background_forecasts.parquet" in source_manifest(SOURCE).get("artifacts",{}):
            source_baseline = pd.read_parquet(source_artifact(SOURCE,"data/background_forecasts.parquet"))
    device = collection_device()
    print("Chronos device:",device,"| reusable backgrounds per generator:",SOURCE["n_bg"] if SOURCE else 0)
    parts,baseline_parts = [],[]
    new_tone_forecasts,new_background_forecasts = 0,0
    start = time.time()
    for P,S in MODELS:
        tag = pl.model_tag(P,S)
        base = design[design.model.eq(tag)].reset_index(drop=True)
        probe = None
        try:
            for generator in GENERATORS:
                for first in range(0,N_BG,BG_PER_SHARD):
                    arm_path = DATA_ROOT/"raw"/f"arms_{tag}_{generator}_{first:03d}.parquet"
                    cal_path = DATA_ROOT/"raw"/f"baseline_{tag}_{generator}_{first:03d}.parquet"
                    arms_done,cal_done = valid_artifact(arm_path),valid_artifact(cal_path)
                    if arms_done and cal_done:
                        parts.append(pd.read_parquet(arm_path)); baseline_parts.append(pd.read_parquet(cal_path))
                        continue
                    block_arms,block_cal = [],[]
                    for bg_id in range(first,min(first+BG_PER_SHARD,N_BG)):
                        bg = canonical_background(generator,bg_id)
                        reusable = SOURCE is not None and bg_id<SOURCE["n_bg"]
                        if not arms_done:
                            if reusable:
                                meta = source_arms[source_arms.model.eq(tag)&source_arms.generator.eq(generator)&source_arms.bg_id.eq(bg_id)].copy()
                            else:
                                if probe is None:
                                    probe = pl.Probe(P,S,device=device,batch_size=BATCH_SIZE)
                                    if probe.checkpoint_identity!=CHECKPOINT_IDENTITIES[tag]: raise ValueError("Checkpoint changed")
                                meta = base.copy()
                                meta["generator"],meta["bg_id"] = generator,bg_id
                                signals = np.stack([pl.build_context(bg,row.f,row.phase,pl.CANON_LEN) for row in base.itertuples(index=False)])
                                predicted = probe.forecast(signals[:,:pl.CTX])
                                peaks,truth_peaks = spectral_peaks(predicted),spectral_peaks(signals[:,pl.CTX:])
                                meta["h"],meta["h_truth"] = hit_values(peaks,meta.f),hit_values(truth_peaks,meta.f)
                                meta["h_blind"] = hit_values(np.repeat(spectral_peaks(bg[pl.CTX:]),len(meta),axis=0),meta.f)
                                for j in range(TOP_K):
                                    meta[f"f_hat_{j+1}"],meta[f"truth_f_hat_{j+1}"] = peaks[:,j],truth_peaks[:,j]
                                new_tone_forecasts += len(meta)
                            block_arms.append(meta)
                        if not cal_done:
                            reused_cal = None
                            if reusable and source_baseline is not None:
                                selected = source_baseline[source_baseline.model.eq(tag)&source_baseline.generator.eq(generator)&source_baseline.bg_id.eq(bg_id)]
                                if len(selected)!=1: raise ValueError("Source baseline row missing or duplicate")
                                reused_cal = selected.iloc[0].to_dict()
                            if reused_cal is None:
                                if probe is None:
                                    probe = pl.Probe(P,S,device=device,batch_size=BATCH_SIZE)
                                    if probe.checkpoint_identity!=CHECKPOINT_IDENTITIES[tag]: raise ValueError("Checkpoint changed")
                                # Critical: pass the same background context through Chronos, with no tone.
                                predicted_bg = np.asarray(probe.forecast(bg[None,:pl.CTX]),float)
                                peaks_bg = spectral_peaks(predicted_bg)[0]
                                reused_cal = dict(model=tag,generator=generator,bg_id=bg_id)
                                reused_cal.update({f"forecast_{i}":float(value) for i,value in enumerate(predicted_bg[0])})
                                reused_cal.update({f"background_f_hat_{j+1}":float(value) for j,value in enumerate(peaks_bg)})
                                new_background_forecasts += 1
                            block_cal.append(reused_cal)
                    if not arms_done: save_table(arm_path,pd.concat(block_arms,ignore_index=True))
                    if not cal_done: save_table(cal_path,pd.DataFrame(block_cal))
                    parts.append(pd.read_parquet(arm_path)); baseline_parts.append(pd.read_parquet(cal_path))
                    print(f"{tag} {generator} bg {first}:{min(first+BG_PER_SHARD,N_BG)} saved; {(time.time()-start)/60:.1f} min")
        finally:
            if probe is not None: probe.close()
            gc.collect()
    arms,calibration = pd.concat(parts,ignore_index=True),pd.concat(baseline_parts,ignore_index=True)
    validate_arms(arms,design)
    validate_calibration(calibration)
    arms = attach_calibration(arms,calibration)
    save_table(baseline_path,calibration)
    save_table(merged_path,arms)
    print("New tone/background-only forecasts:",new_tone_forecasts,new_background_forecasts)
    return arms,calibration

def calibration_summary(arms,columns):
    result = arms.groupby(columns,observed=True).agg(
        n=("h","size"),forecast_hit=("h","mean"),true_future_hit=("h_truth","mean"),
        background_true_future_hit=("h_blind","mean"),background_forecast_hit=("h_background_forecast","mean")).reset_index()
    result["tone_minus_background_forecast"] = result.forecast_hit-result.background_forecast_hit
    result["truth_minus_background_truth"] = result.true_future_hit-result.background_true_future_hit
    return result


### 2.2 Scientific model and exact change of coordinates

For trial i, h_i is 1 when at least one of the three retained spectral peaks lies within +/-1 Hz of its own injected frequency. There is no amplitude-recovery threshold. Identical predictor rows are pooled exactly: y_g ~ Binomial(n_g,p_g), equivalent to their conditionally independent Bernoulli trials.

logit(p_i) = beta_c + gamma_c L_i + kappa_c D_i + u_h + v_b,
where L=1 for lock and 0 for controls, D=-1/0/+1 for lo/lock/hi.

The following are the unchanged original priors (second Normal parameter below is standard deviation):

- beta_c ~ Normal(beta_bar + delta_O O_c* + delta_P logP_c*, tau).
- beta_bar ~ StudentT(nu=4, mu=0, sigma=1.5); delta_O, delta_P ~ StudentT(4,0,0.5).
- gamma_c ~ Normal(gamma_bar,sigma_gamma); kappa_c ~ Normal(kappa_bar,sigma_kappa).
- gamma_bar, kappa_bar ~ StudentT(4,0,0.5).
- tau, sigma_gamma, sigma_kappa, sigma_harm, sigma_bg ~ HalfStudentT(4, sigma=0.5).
- u ~ ZeroSumNormal(sigma_harm); v ~ ZeroSumNormal(sigma_bg), with independent frequency/background vectors.
- O* = (O - mean(O))/0.5; logP* = log(P)-mean(log(P)), over the selected geometries.

For the sampler only, let r be 64 Hz, w_h=u_h-u_r and a_c=beta_c+u_r. Then w_r=0, u_h=w_h-mean(w), beta_c=a_c+mean(w), and eta_i=a_c+w_h+gamma_c L_i+kappa_c D_i+v_b. The non-reference w vector has distribution MVN(0, sigma_harm^2 (I+11')). The original Normal density of beta is evaluated at a+mean(w); this translation has unit conditional Jacobian. Thus the scientific joint prior and likelihood are preserved. `a` is not assigned a new independent flat scientific prior: its Flat node is paired with that exact proper conditional density. Generic prior-predictive sampling of the Flat node is unsupported; use the original generative model for prior draws.

gamma_design = mean(gamma_c), equally weighted over 15 geometries; OR=exp(gamma_design) compares lock odds with the geometric mean of lo/hi odds. Support: P(OR<0.8)>=0.95. Practical equivalence: P(1/1.1<OR<1.1)>=0.95. Opposite direction: P(OR>1.25)>=0.95. Otherwise inconclusive. These are conditional odds contrasts, not marginal probability ratios or amplitude recovery.

Gates: every retained parameter finite, R-hat<1.01, bulk and tail ESS>400 in PILOT and >1000 in FULL, zero post-warmup divergences; both known-truth recoveries have >=80% 95%-interval coverage and include the headline truth; >=90% PPC coverage in each of three check sets. FULL additionally requires all prior/link sensitivity fits to converge and probability spreads <=0.10. Failed or incomplete inference is NOT REPORTABLE.


In [ ]:
def _codes(series):
    """Integer codes plus the ordered level names, for PyMC `coords`."""
    codes, levels = pd.factorize(series)
    return np.asarray(codes), list(map(str, levels))

def _overlap_scaled(df: pd.DataFrame, cfg_levels: list[str]) -> np.ndarray:
    """Centred, scaled patch overlap O = (P-S)/P; one unit = 0.5 of overlap."""
    O = df.groupby("model")["overlap"].first().reindex(cfg_levels).to_numpy(float)
    return (O - O.mean()) / 0.5

def _logP_centred(df: pd.DataFrame, cfg_levels: list[str]) -> np.ndarray:
    """Centred log patch size.

    Deliverable 3, H1: "The overlap enters as a ratio and the patch size as log P, because the
    patch grid has spacing fs/P: equal steps in log P are then equal ratios of spacing." On a raw-P
    scale one coefficient would make 8->16 and 16->24 the same change, which the geometry does not.
    """
    Pv = df.groupby("model")["P"].first().reindex(cfg_levels).to_numpy(float)
    lp = np.log(Pv)
    return lp - lp.mean()

def model_A2_original(frame, scale=PRIOR_SCALE, baseline_scale=BASELINE_SCALE, nu=NU,
             link="logit", encoding="counts"):
    """Bernoulli A2; counts pool only rows with identical full predictor vectors.

    eta = beta_config + gamma_config*is_lock + kappa_config*side + u_harm + u_bg
    side = -1 (lo), 0 (lock), +1 (hi).
    gamma_config is the lock log odds minus the midpoint of the two control log odds.
    gamma_design is the equally weighted mean over the 15 observed configurations.
    """
    if link not in {"logit","probit"} or encoding not in {"counts","bernoulli"}:
        raise ValueError("Unknown link or likelihood encoding")
    ci,cl = _codes(frame.model)
    hi,hl = _codes(frame.f_lock.round(3).astype(str))
    bi,bl = _codes(frame.generator+"#"+frame.bg_id.astype(str))
    expected_side = frame.role.map({"lo":-1,"lock":0,"hi":1})
    if expected_side.isna().any() or not frame.is_lock.eq(frame.role.eq("lock").astype(int)).all():
        raise ValueError("Invalid role/lock encoding")
    if "side" in frame and not frame.side.eq(expected_side).all():
        raise ValueError("Side differs from the registered role encoding")
    y = frame["hits" if encoding=="counts" else "h"].to_numpy(float)
    n = frame.n_trials.to_numpy(float) if encoding=="counts" else np.ones(len(frame))
    if not (len(frame) and np.isfinite(y).all() and np.isfinite(n).all()
            and np.equal(y,np.floor(y)).all() and np.equal(n,np.floor(n)).all()
            and (n>=1).all() and (y>=0).all() and (y<=n).all()):
        raise ValueError("Invalid finite integer counts")
    coords = dict(config=cl,harmonic=hl,background=bl,obs=np.arange(len(frame)))
    with pm.Model(coords=coords) as model:
        beta_bar = pm.StudentT("beta_bar",nu=nu,mu=0,sigma=baseline_scale)
        delta_O = pm.StudentT("delta_O",nu=nu,mu=0,sigma=scale)
        delta_P = pm.StudentT("delta_P",nu=nu,mu=0,sigma=scale)
        tau = pm.HalfStudentT("tau",nu=nu,sigma=scale)
        beta = pm.Normal("beta",mu=beta_bar+delta_O*_overlap_scaled(frame,cl)+delta_P*_logP_centred(frame,cl),sigma=tau,dims="config")
        gamma_bar = pm.StudentT("gamma_bar",nu=nu,mu=0,sigma=scale)
        sigma_gamma = pm.HalfStudentT("sigma_gamma",nu=nu,sigma=scale)
        gamma = pm.Normal("gamma_config",mu=gamma_bar,sigma=sigma_gamma,dims="config")
        kappa_bar = pm.StudentT("kappa_bar",nu=nu,mu=0,sigma=scale)
        sigma_kappa = pm.HalfStudentT("sigma_kappa",nu=nu,sigma=scale)
        kappa = pm.Normal("kappa_config",mu=kappa_bar,sigma=sigma_kappa,dims="config")
        sigma_harm = pm.HalfStudentT("sigma_harm",nu=nu,sigma=scale)
        harm = pm.ZeroSumNormal("u_harm",sigma=sigma_harm,dims="harmonic")
        sigma_bg = pm.HalfStudentT("sigma_bg",nu=nu,sigma=scale)
        bg = pm.ZeroSumNormal("u_bg",sigma=sigma_bg,dims="background")
        eta = beta[ci]+gamma[ci]*frame.is_lock.to_numpy(float)+kappa[ci]*expected_side.to_numpy(float)+harm[hi]+bg[bi]
        parameter = {"logit_p":eta} if link=="logit" else {"p":pm.math.clip(.5*(1+pm.math.erf(eta/np.sqrt(2.))),1e-9,1-1e-9)}
        if encoding=="counts":
            pm.Binomial("hits",n=n.astype(int),observed=y.astype(int),dims="obs",**parameter)
        else:
            pm.Bernoulli("h",observed=y.astype(int),dims="obs",**parameter)
        if link=="logit":
            pm.Deterministic("gamma_design",pm.math.mean(gamma))
            pm.Deterministic("odds_ratio_design",pm.math.exp(pm.math.mean(gamma)))
    return model

def aggregate_trials(frame):
    keys = ["model","P","S","overlap","generator","bg_id","f_lock","role","is_lock"]
    result = frame.groupby(keys,observed=True,sort=True).agg(
        hits=("h","sum"),n_trials=("h","size"),truth_hits=("h_truth","sum"),
        background_truth_hits=("h_blind","sum"),background_forecast_hits=("h_background_forecast","sum")).reset_index()
    result["side"] = result.role.map({"lo":-1,"lock":0,"hi":1}).astype(int)
    for name in ("hits","n_trials","truth_hits","background_truth_hits","background_forecast_hits"):
        result[name] = result[name].astype(np.int64)
    assert result.n_trials.sum()==len(frame) and result.hits.sum()==frame.h.sum()
    return result


def model_A2(frame, scale=PRIOR_SCALE, baseline_scale=BASELINE_SCALE, nu=NU,
             link="logit", encoding="counts"):
    """Exact A2 change of coordinates, retaining the original joint prior.

    v_h = u_h - u_reference, a_c = beta_c + u_reference.
    v_reference=0, u=v-mean(v), beta=a+mean(v).
    Non-reference v ~ MVN(0, sigma_harm**2 * (I + 11')).
    The Flat a plus beta_prior Potential is the proper original conditional prior
    expressed in a translated coordinate, with unit conditional Jacobian.
    """
    import pytensor.tensor as pt
    if link not in {"logit", "probit"} or encoding not in {"counts", "bernoulli"}:
        raise ValueError("Unknown link or encoding")
    ci, cl = _codes(frame.model)
    hi, hl = _codes(frame.f_lock.round(3).astype(str))
    bi, bl = _codes(frame.generator+"#"+frame.bg_id.astype(str))
    if len(cl) < 2 or len(hl) < 2 or len(bl) < 2:
        raise ValueError("A2 requires multiple geometries, frequencies and backgrounds")
    reference = np.flatnonzero(np.isclose(np.asarray(hl,float), REFERENCE_HZ))
    if len(reference) != 1:
        raise ValueError("The registered reference frequency is absent/ambiguous")
    reference = int(reference[0])
    free = np.array([i for i in range(len(hl)) if i != reference], dtype=int)
    side = frame.role.map({"lo":-1,"lock":0,"hi":1})
    if side.isna().any() or not frame.is_lock.eq(frame.role.eq("lock").astype(int)).all():
        raise ValueError("Invalid role/lock encoding")
    if "side" in frame and not frame.side.eq(side).all():
        raise ValueError("Invalid side encoding")
    y = frame["hits" if encoding == "counts" else "h"].to_numpy(float)
    n = frame.n_trials.to_numpy(float) if encoding == "counts" else np.ones(len(frame))
    if not (len(frame) and np.isfinite(y).all() and np.isfinite(n).all()
            and np.equal(y,np.floor(y)).all() and np.equal(n,np.floor(n)).all()
            and (n>=1).all() and (y>=0).all() and (y<=n).all()):
        raise ValueError("Invalid finite integer counts")
    coords = dict(config=cl,harmonic=hl,background=bl,obs=np.arange(len(frame)),
                  harmonic_difference=[hl[i] for i in free])
    with pm.Model(coords=coords) as model:
        beta_bar = pm.StudentT("beta_bar",nu=nu,mu=0,sigma=baseline_scale)
        delta_O = pm.StudentT("delta_O",nu=nu,mu=0,sigma=scale)
        delta_P = pm.StudentT("delta_P",nu=nu,mu=0,sigma=scale)
        tau = pm.HalfStudentT("tau",nu=nu,sigma=scale)
        sigma_harm = pm.HalfStudentT("sigma_harm",nu=nu,sigma=scale)
        chol = np.linalg.cholesky(np.eye(len(free))+np.ones((len(free),len(free))))
        differences = pm.MvNormal("frequency_difference",mu=np.zeros(len(free)),
                                  chol=sigma_harm*chol,dims="harmonic_difference")
        v = pt.set_subtensor(pt.zeros(len(hl))[free], differences)
        v_mean = pt.mean(v)
        harm = pm.Deterministic("u_harm",v-v_mean,dims="harmonic")
        a = pm.Flat("baseline_at_reference",dims="config",initval=np.zeros(len(cl)))
        beta = pm.Deterministic("beta",a+v_mean,dims="config")
        beta_mu = beta_bar+delta_O*_overlap_scaled(frame,cl)+delta_P*_logP_centred(frame,cl)
        pm.Potential("beta_prior",pm.logp(pm.Normal.dist(mu=beta_mu,sigma=tau),beta).sum())
        gamma_bar = pm.StudentT("gamma_bar",nu=nu,mu=0,sigma=scale)
        sigma_gamma = pm.HalfStudentT("sigma_gamma",nu=nu,sigma=scale)
        gamma = pm.Normal("gamma_config",mu=gamma_bar,sigma=sigma_gamma,dims="config")
        kappa_bar = pm.StudentT("kappa_bar",nu=nu,mu=0,sigma=scale)
        sigma_kappa = pm.HalfStudentT("sigma_kappa",nu=nu,sigma=scale)
        kappa = pm.Normal("kappa_config",mu=kappa_bar,sigma=sigma_kappa,dims="config")
        sigma_bg = pm.HalfStudentT("sigma_bg",nu=nu,sigma=scale)
        bg = pm.ZeroSumNormal("u_bg",sigma=sigma_bg,dims="background")
        eta = a[ci]+v[hi]+gamma[ci]*frame.is_lock.to_numpy(float)+kappa[ci]*side.to_numpy(float)+bg[bi]
        parameter = {"logit_p":eta} if link=="logit" else {"p":pm.math.clip(.5*(1+pm.math.erf(eta/np.sqrt(2.))),1e-9,1-1e-9)}
        if encoding=="counts":
            pm.Binomial("hits",n=n.astype(int),observed=y.astype(int),dims="obs",**parameter)
        else:
            pm.Bernoulli("h",observed=y.astype(int),dims="obs",**parameter)
        if link=="logit":
            pm.Deterministic("gamma_design",pm.math.mean(gamma))
            pm.Deterministic("odds_ratio_design",pm.math.exp(pm.math.mean(gamma)))
    return model


### 2.3 Inference, complete-chain checkpoints and reporting

Each chain has its own seed, full warmup and `*.chain_XX.nc` artifact. After interruption, rerun with the same settings: completed chains reload; missing chains run. Zero-length, partial, non-finite, wrong-coordinate and wrong-fingerprint files cannot become a completed posterior. An interrupted returned trace is retained under `checkpoints/partial/` for inspection only. It is never concatenated into the final inference.

All requested chains are combined before R-hat/ESS and final gates. No chain is discarded for an unfavorable diagnostic. If a legacy empty checkpoint is detected, it is preserved and rejected with an explicit error; select a new label or run namespace rather than deleting measurements.


In [ ]:
def _group_names(idata):
    names = idata.groups
    return list(names() if callable(names) else (str(x).strip('/') for x in names if str(x).strip('/')))


def _dataset(idata, name):
    group = idata[name]
    return group if isinstance(group, xr.Dataset) else group.to_dataset()


def _make_idata(datasets, attrs=None):
    # PyMC 6 / ArviZ 1 use DataTree; earlier releases use InferenceData.
    if hasattr(az, 'InferenceData'):
        return az.InferenceData(attrs=dict(attrs or {}), **datasets)
    result = xr.DataTree.from_dict({'/': xr.Dataset(attrs=dict(attrs or {})),
                                    **{f'/{k}': v for k, v in datasets.items()}})
    return result


def _load_idata(path):
    result = az.from_netcdf(path) if hasattr(az, 'from_netcdf') else xr.open_datatree(path, engine='h5netcdf')
    names = _group_names(result)
    try:
        datasets = {name:_dataset(result, name).load() for name in names}
        attrs = dict(result.attrs)
    finally:
        # Legacy InferenceData.load()/close() default to a copy. Close the actual
        # datasets, otherwise Windows can keep checkpoint files locked.
        for name in names:
            _dataset(result, name).close()
        if isinstance(result, xr.DataTree):
            result.close()
    return _make_idata(datasets, attrs)


def validate_posterior(idata, draws=None, chains=None, expected_coords=None, link=None):
    """Reject incomplete output, independently of its convergence diagnostics."""
    if not {'posterior', 'sample_stats'}.issubset(_group_names(idata)):
        raise ValueError('Missing posterior or sample_stats')
    posterior, stats = _dataset(idata, 'posterior'), _dataset(idata, 'sample_stats')
    for group in (posterior, stats):
        for dim, expected in (('chain', chains), ('draw', draws)):
            count = group.sizes.get(dim, 0)
            if count <= 0 or (expected is not None and count != expected):
                raise ValueError(f'Incomplete posterior: {dim}={count}, expected {expected or "positive"}')
            if dim not in group.coords or len(np.unique(group[dim].values)) != count:
                raise ValueError(f'Missing/duplicate {dim} coordinates')
    for dim in ('chain', 'draw'):
        if not np.array_equal(posterior[dim], stats[dim]):
            raise ValueError(f'Posterior/sample_stats {dim} mismatch')
    required = {'beta_bar','delta_O','delta_P','tau','beta','gamma_bar','sigma_gamma',
                'gamma_config','kappa_bar','sigma_kappa','kappa_config','sigma_harm','u_harm','sigma_bg','u_bg'}
    if link == 'logit':
        required |= {'gamma_design','odds_ratio_design'}
    if not required.issubset(posterior.data_vars):
        raise ValueError(f'Missing posterior variables: {required-set(posterior.data_vars)}')
    for name, array in posterior.data_vars.items():
        if not {'chain','draw'}.issubset(array.dims) or not np.isfinite(array.values).all():
            raise ValueError(f'Invalid posterior array: {name}')
    if 'diverging' not in stats or not np.isin(stats.diverging.values, [False, True]).all():
        raise ValueError('Missing/invalid divergence flags')
    for name in ('lp','energy'):
        if name not in stats or not np.isfinite(stats[name].values).all():
            raise ValueError(f'Missing/non-finite sampler statistic: {name}')
    for name, values in (expected_coords or {}).items():
        if name not in posterior.coords or list(map(str, posterior[name].values)) != list(map(str, values)):
            raise ValueError(f'Posterior {name} levels differ from the input data')
    if 'log_likelihood' in _group_names(idata) or 'p' in posterior or 'obs' in posterior.dims:
        raise ValueError('Observation-sized posterior output unexpectedly retained')
    return dict(posterior.sizes)


def _chain_idata(idata, chain_id):
    datasets = {}
    for name in _group_names(idata):
        ds = _dataset(idata, name)
        datasets[name] = ds.assign_coords(chain=[chain_id]) if 'chain' in ds.dims else ds
    return _make_idata(datasets, idata.attrs)


def _combine_chains(chains):
    names = _group_names(chains[0])
    if any(set(_group_names(c)) != set(names) for c in chains):
        raise ValueError('Chain groups differ')
    datasets = {}
    for name in names:
        pieces = [_dataset(c, name) for c in chains]
        if 'chain' in pieces[0].dims:
            datasets[name] = xr.concat(pieces, dim='chain', join='exact', compat='equals', coords='minimal')
        else:
            if any(not pieces[0].equals(p) for p in pieces[1:]):
                raise ValueError(f'Non-sampled data differ across chains: {name}')
            datasets[name] = pieces[0]
    return _make_idata(datasets)


def fit_or_load(label, frame, scale=PRIOR_SCALE, link='logit', draws=None, tune=None):
    draws = DRAWS if draws is None else int(draws)
    tune = TUNE if tune is None else int(tune)
    if draws < 1 or tune < 1 or CHAINS < 2:
        raise ValueError('Positive retained draws/warmup and at least two chains are required')
    path = CHECKPOINT_ROOT/f'{label}.nc'
    fields = ['model','P','S','overlap','generator','bg_id','f_lock','role','is_lock','side','hits','n_trials']
    data_hash = hashlib.sha256(pd.util.hash_pandas_object(frame[fields], index=False).to_numpy().tobytes()).hexdigest()
    label_seed = int(hashlib.sha256(label.encode()).hexdigest()[:8], 16)
    seeds = np.random.SeedSequence([SEED, label_seed]).generate_state(CHAINS).tolist()
    expected_coords = dict(config=_codes(frame.model)[1], harmonic=_codes(frame.f_lock.round(3).astype(str))[1],
                           background=_codes(frame.generator+'#'+frame.bg_id.astype(str))[1])
    fit_spec = dict(model=MODEL_VERSION, parameterization='reference64-v1', reference_hz=REFERENCE_HZ,
                    data_sha256=data_hash, scale=scale, baseline_scale=BASELINE_SCALE, nu=NU, link=link,
                    draws=draws, tune=tune, chains=CHAINS, target_accept=TARGET_ACCEPT, seeds=seeds,
                    backend='pymc', init=NUTS_INIT, analysis=ANALYSIS_FINGERPRINT)
    fit_hash = cp.fingerprint(fit_spec)

    def load_complete(target, n_chains, chain_id=None):
        if not valid_artifact(target):
            return None
        result = _load_idata(target)
        if result.attrs.get('fit_fingerprint') != fit_hash or result.attrs.get('complete') != 'yes':
            raise ValueError(f'Checkpoint settings/completeness differ: {target}. Preserve it and choose a new label/RUN_ID.')
        try:
            validate_posterior(result, draws, n_chains, expected_coords, link)
        except ValueError as exc:
            raise ValueError(f'Unusable checkpoint {target}: {exc}. It will not be treated as a completed fit.') from exc
        expected_chain_ids = [chain_id] if chain_id is not None else list(range(CHAINS))
        if list(_dataset(result,'posterior').chain.values) != expected_chain_ids:
            raise ValueError(f'Wrong chain identity: {target}')
        if chain_id is not None and int(result.attrs.get('chain_seed', -1)) != seeds[chain_id]:
            raise ValueError(f'Wrong chain seed: {target}')
        return result

    cached = load_complete(path, CHAINS)
    if cached is not None:
        print(f'{label}: reloaded {CHAINS} complete chains x {draws} draws', flush=True)
        return cached
    unit = 1.6 if link == 'probit' else 1.
    model = None
    completed = []
    start = time.monotonic()
    print(f'{label}: {len(frame):,} groups / {frame.n_trials.sum():,} trials; '
          f'{CHAINS} sequential chains, each {tune} warmup + {draws} retained draws; {NUTS_INIT}', flush=True)
    for chain_id, seed in enumerate(seeds):
        chain_path = CHECKPOINT_ROOT/f'{label}.chain_{chain_id:02d}.nc'
        result = load_complete(chain_path, 1, chain_id)
        if result is not None:
            print(f'  Chain {chain_id+1}/{CHAINS}: complete checkpoint reloaded', flush=True)
            completed.append(result)
            continue
        if model is None:
            model = model_A2(frame, scale=scale/unit, baseline_scale=BASELINE_SCALE/unit, link=link)
            if not np.isfinite(model.compile_logp()(model.initial_point())):
                raise ValueError('Non-finite initial log probability')
        chain_start = time.monotonic()
        print(f'  Chain {chain_id+1}/{CHAINS}: starting, seed={seed}', flush=True)
        def progress(trace, draw):
            iteration = int(draw.draw_idx)+1
            if iteration % PROGRESS_EVERY == 0 or iteration == tune+draws:
                phase = 'warmup' if draw.tuning else 'posterior'
                print(f'  Chain {chain_id+1}/{CHAINS} {phase}: {iteration}/{tune+draws}; '
                      f'elapsed {(time.monotonic()-chain_start)/60:.1f} min', flush=True)
        with model:
            result = pm.sample(draws=draws, tune=tune, chains=1, cores=1, random_seed=int(seed),
                               target_accept=TARGET_ACCEPT, init=NUTS_INIT, nuts_sampler='pymc',
                               callback=progress, progressbar=False, return_inferencedata=True,
                               compute_convergence_checks=False, discard_tuned_samples=True,
                               idata_kwargs={'log_likelihood':False})
        try:
            validate_posterior(result, draws, 1, expected_coords, link)
        except ValueError as exc:
            partial = CHECKPOINT_ROOT/'partial'/f'{label}.chain_{chain_id:02d}.{time.time_ns()}.nc'
            partial.parent.mkdir(exist_ok=True)
            cp.atomic_netcdf(partial, result)
            raise RuntimeError(f'Chain {chain_id+1} returned incomplete output: {exc}. '
                               f'Inspection copy: {partial}. Rerun this cell with unchanged settings; '
                               'completed chains reload, this chain restarts its warmup.') from exc
        result = _chain_idata(result, chain_id)
        result.attrs.update(fit_fingerprint=fit_hash, model_version=MODEL_VERSION, complete='yes',
                            chain_seed=int(seed), fit_spec_json=cp.canonical_json(fit_spec))
        cp.atomic_netcdf(chain_path, result)
        # Verify the serialized content before recording a completed artifact.
        roundtrip = _load_idata(chain_path)
        validate_posterior(roundtrip, draws, 1, expected_coords, link)
        record_artifact(chain_path)
        completed.append(roundtrip)
        print(f'  Saved complete chain {chain_id+1}/{CHAINS}', flush=True)
    combined = _combine_chains(completed)
    validate_posterior(combined, draws, CHAINS, expected_coords, link)
    combined.attrs.update(fit_fingerprint=fit_hash, model_version=MODEL_VERSION, complete='yes',
                          fit_spec_json=cp.canonical_json(fit_spec))
    cp.atomic_netcdf(path, combined)
    validate_posterior(_load_idata(path), draws, CHAINS, expected_coords, link)
    record_artifact(path)
    print(f'Saved {path.name}; this invocation took {(time.monotonic()-start)/60:.1f} minutes', flush=True)
    return combined


def diagnostics_for(idata,ess_min):
    validate_posterior(idata)
    table = az.summary(idata,kind="diagnostics",round_to="none")
    table = table[["r_hat","ess_bulk","ess_tail"]].apply(pd.to_numeric,errors="coerce")
    result = dict(max_rhat=float(table.r_hat.max()),min_ess_bulk=float(table.ess_bulk.min()),
                  min_ess_tail=float(table.ess_tail.min()),divergences=int(np.asarray(idata.sample_stats["diverging"]).sum()))
    result["passed"] = bool(np.isfinite(table.to_numpy(float)).all() and (table.r_hat<RHAT_MAX).all()
                            and (table.ess_bulk>ess_min).all() and (table.ess_tail>ess_min).all() and result["divergences"]==0)
    return result,table

def posterior_array(idata,name,dimension=None):
    values = np.asarray(idata.posterior[name].transpose("chain","draw",*([dimension] if dimension else [])),float)
    return values.reshape(-1,values.shape[-1]) if dimension else values.ravel()

def posterior_components(idata,frame):
    def codes(values,dimension):
        lookup = {str(value):i for i,value in enumerate(idata.posterior.coords[dimension].values)}
        result = np.array([lookup.get(str(value),-1) for value in values],int)
        if (result<0).any(): raise ValueError(f"Unknown posterior {dimension} level")
        return result
    return dict(beta=posterior_array(idata,"beta","config"),gamma=posterior_array(idata,"gamma_config","config"),
                kappa=posterior_array(idata,"kappa_config","config"),harm=posterior_array(idata,"u_harm","harmonic"),
                bg=posterior_array(idata,"u_bg","background"),ci=codes(frame.model,"config"),
                hi=codes(frame.f_lock.round(3).astype(str),"harmonic"),bi=codes(frame.generator+"#"+frame.bg_id.astype(str),"background"))

def probability(eta,link):
    return expit(eta) if link=="logit" else np.clip(ndtr(eta),1e-9,1-1e-9)

def effect_summary(values):
    values = np.asarray(values, dtype=float)
    if values.size == 0 or not np.isfinite(values).all():
        raise ValueError("Posterior samples are empty or non-finite; effect is not reportable")
    low,median,high = np.quantile(values,[.025,.5,.975])
    return dict(median_log_or=float(median),eti_low_log_or=float(low),eti_high_log_or=float(high),
                median_odds_ratio=float(np.exp(median)),p_support=float(np.mean(values<SUPPORT_LOG_OR)),
                p_equivalence=float(np.mean(np.abs(values)<ROPE_LOG_OR)),
                p_opposite=float(np.mean(values>-SUPPORT_LOG_OR)),p_positive=float(np.mean(values>0)))

def comparable_effects(idata,frame,link="logit"):
    """Per-geometry log odds against the midpoint of the two control log odds."""
    if link=="logit":
        return posterior_array(idata,"gamma_config","config")
    lock_rows = frame[frame.role.eq("lock")].reset_index(drop=True)
    arrays = posterior_components(idata,lock_rows)
    n = lock_rows.n_trials.to_numpy(float)
    weights = n/np.bincount(arrays["ci"],weights=n)[arrays["ci"]]
    selected = np.linspace(0,len(arrays["beta"])-1,min(EFFECT_DRAWS,len(arrays["beta"]))).astype(int)
    results = []
    for i in selected:
        base = arrays["beta"][i,arrays["ci"]]+arrays["harm"][i,arrays["hi"]]+arrays["bg"][i,arrays["bi"]]
        gamma,kappa = arrays["gamma"][i,arrays["ci"]],arrays["kappa"][i,arrays["ci"]]
        difference = logit(probability(base+gamma,link))-.5*(logit(probability(base-kappa,link))+logit(probability(base+kappa,link)))
        results.append(np.bincount(arrays["ci"],weights=weights*difference,minlength=arrays["beta"].shape[1]))
    return np.asarray(results)

def geometry_effect_table(idata,frame):
    gamma,kappa = posterior_array(idata,"gamma_config","config"),posterior_array(idata,"kappa_config","config")
    rows = []
    for i,name in enumerate(idata.posterior.coords["config"].values):
        row = dict(model=str(name),**effect_summary(gamma[:,i]))
        for label,values in (("lock_vs_lo",gamma[:,i]+kappa[:,i]),("lock_vs_hi",gamma[:,i]-kappa[:,i]),("side",kappa[:,i])):
            lo,med,hi = np.quantile(values,[.025,.5,.975])
            row.update({label+"_median_log_or":float(med),label+"_low_log_or":float(lo),label+"_high_log_or":float(hi)})
        rows.append(row)
    return pd.DataFrame(rows)

def predictive_check(idata,frame,seed=SEED+600):
    arrays,rng = posterior_components(idata,frame),np.random.default_rng(seed)
    selected = np.linspace(0,len(arrays["beta"])-1,min(PPC_DRAWS,len(arrays["beta"]))).astype(int)
    specs = []
    for check_set,arm_column in (("A1_comparable","is_lock"),("separate_roles","role")):
        for dimension in ("model","generator"):
            for (level,arm),positions in frame.groupby([dimension,arm_column],observed=True).indices.items():
                specs.append(dict(check_set=check_set,dimension=dimension,level=str(level),arm=str(arm),
                                  key=f"{dimension}={level}|{arm_column}={arm}",positions=np.asarray(positions,int)))
    simulated_rates = [[] for _ in specs]
    n = frame.n_trials.to_numpy(int)
    for i in selected:
        eta = (arrays["beta"][i,arrays["ci"]]+arrays["gamma"][i,arrays["ci"]]*frame.is_lock.to_numpy(float)
               +arrays["kappa"][i,arrays["ci"]]*frame.side.to_numpy(float)
               +arrays["harm"][i,arrays["hi"]]+arrays["bg"][i,arrays["bi"]])
        replicate = rng.binomial(n,expit(eta))
        for samples,spec in zip(simulated_rates,specs):
            pos = spec["positions"]
            samples.append(replicate[pos].sum()/n[pos].sum())
    rows = []
    for samples,spec in zip(simulated_rates,specs):
        pos = spec["positions"]
        observed = float(frame.hits.to_numpy()[pos].sum()/n[pos].sum())
        low,high = np.quantile(samples,[.025,.975])
        rows.append({**{k:v for k,v in spec.items() if k!="positions"},"n_trials":int(n[pos].sum()),
                     "observed_rate":observed,"rep_low":float(low),"rep_high":float(high),"passed":bool(low<=observed<=high)})
    return pd.DataFrame(rows)

def comparison_to_A1(ppc):
    if SOURCE is None or SOURCE["kind"]!="A-bernoulli-centred-zerosum-exact-counts-v1" or SOURCE["source_n_bg"]!=N_BG:
        return pd.DataFrame()
    manifest = source_manifest(SOURCE)
    if "primary_ppc.parquet" not in manifest.get("artifacts",{}):
        return pd.DataFrame()
    old = pd.read_parquet(source_artifact(SOURCE,"primary_ppc.parquet"))
    keys = []
    for value in old.stratum:
        match = re.fullmatch(r"\(\('(model|generator)', '([^']+)'\), \('is_lock', (?:np\.\w+\()?([01])\)?\)\)",str(value))
        if match is None: raise ValueError(f"Unrecognised A1 PPC key: {value}")
        dimension,level,arm = match.groups()
        keys.append(f"{dimension}={level}|is_lock={arm}")
    old["key"] = keys
    new = ppc[ppc.check_set.eq("A1_comparable")]
    if set(old.key)!=set(new.key) or old.key.duplicated().any(): raise ValueError("A1/A2 PPC strata differ")
    result = new.merge(old[["key","n_trials","observed_rate","rep_low","rep_high","passed"]],on="key",suffixes=("_A2","_A1"),validate="one_to_one")
    if not result.n_trials_A2.eq(result.n_trials_A1).all(): raise ValueError("A1/A2 PPC trial counts differ")
    np.testing.assert_allclose(result.observed_rate_A2,result.observed_rate_A1,atol=1e-12,rtol=0)
    return result

def recovery_data(groups):
    out = groups[groups.bg_id<3].copy().reset_index(drop=True)
    rng = np.random.default_rng(SEED+300)
    ci,cl = _codes(out.model)
    hi,hl = _codes(out.f_lock.round(3).astype(str))
    bi,bl = _codes(out.generator+"#"+out.bg_id.astype(str))
    truth = dict(beta_bar=-.4,delta_O=.2,delta_P=-.15,gamma_bar=-.25,kappa_bar=-.2)
    beta = rng.normal(truth["beta_bar"]+truth["delta_O"]*_overlap_scaled(out,cl)+truth["delta_P"]*_logP_centred(out,cl),.3)
    gamma,kappa = rng.normal(truth["gamma_bar"],.65,len(cl)),rng.normal(truth["kappa_bar"],.3,len(cl))
    harm = rng.normal(0,.4,len(hl)); harm-=harm.mean()
    bg = rng.normal(0,.2,len(bl)); bg-=bg.mean()
    eta = beta[ci]+gamma[ci]*out.is_lock.to_numpy()+kappa[ci]*out.side.to_numpy()+harm[hi]+bg[bi]
    out["hits"] = rng.binomial(out.n_trials.to_numpy(int),expit(eta))
    truth["gamma_design"] = float(gamma.mean())
    rows = [dict(parameter=name,variable=name,coordinate=None,truth=value) for name,value in truth.items()]
    rows += [dict(parameter=f"{name}[{level}]",variable=name,coordinate=level,truth=float(value))
             for name,values in (("gamma_config",gamma),("kappa_config",kappa)) for level,value in zip(cl,values)]
    return out,pd.DataFrame(rows)

def verdict_from(gate_ok,effect):
    if not gate_ok: return "NOT REPORTABLE"
    if effect["p_support"]>=PROB_CUTOFF: return "SUPPORTED"
    if effect["p_equivalence"]>=PROB_CUTOFF: return "PRACTICALLY EQUIVALENT"
    if effect["p_opposite"]>=PROB_CUTOFF: return "OPPOSITE DIRECTION"
    return "INCONCLUSIVE"


def fine_predictive_check(idata, frame, seed=SEED+1600):
    arrays, rng = posterior_components(idata, frame), np.random.default_rng(seed)
    selected = np.linspace(0,len(arrays['beta'])-1,min(PPC_DRAWS,len(arrays['beta']))).astype(int)
    keys = pd.MultiIndex.from_frame(frame[['model','f_lock','role']])
    codes, levels = pd.factorize(keys, sort=True)
    n = frame.n_trials.to_numpy(int)
    totals = np.bincount(codes, weights=n)
    observed = np.bincount(codes, weights=frame.hits.to_numpy())/totals
    replicated = np.empty((len(selected),len(levels)))
    lock, side = frame.is_lock.to_numpy(float), frame.side.to_numpy(float)
    for j,i in enumerate(selected):
        eta = (arrays['beta'][i,arrays['ci']]+arrays['gamma'][i,arrays['ci']]*lock
               +arrays['kappa'][i,arrays['ci']]*side+arrays['harm'][i,arrays['hi']]+arrays['bg'][i,arrays['bi']])
        replicated[j] = np.bincount(codes, weights=rng.binomial(n,expit(eta)))/totals
    low, high = np.quantile(replicated,[.025,.975],axis=0)
    result = levels.to_frame(index=False)
    result.columns = ['model','f_lock','role']
    return result.assign(n_trials=totals.astype(int),observed_rate=observed,rep_low=low,rep_high=high,
                         passed=(low<=observed)&(observed<=high))


def sparse_recovery_data(groups):
    """Fixed challenging truth, chosen before sampling; no tuning to gate outcomes."""
    out = groups[groups.bg_id<10].copy().reset_index(drop=True)
    rng = np.random.default_rng(SEED+1300)
    ci,cl = _codes(out.model)
    hi,hl = _codes(out.f_lock.round(3).astype(str))
    bi,bl = _codes(out.generator+'#'+out.bg_id.astype(str))
    truth = dict(beta_bar=-8.,delta_O=.2,delta_P=-.15,tau=.3,gamma_bar=.8,sigma_gamma=.65,
                 kappa_bar=-.2,sigma_kappa=.3,sigma_harm=5.,sigma_bg=.4)
    beta = rng.normal(truth['beta_bar']+truth['delta_O']*_overlap_scaled(out,cl)
                      +truth['delta_P']*_logP_centred(out,cl),truth['tau'])
    gamma = rng.normal(truth['gamma_bar'],truth['sigma_gamma'],len(cl))
    kappa = rng.normal(truth['kappa_bar'],truth['sigma_kappa'],len(cl))
    harm = rng.normal(0,truth['sigma_harm'],len(hl)); harm -= harm.mean()
    bg = rng.normal(0,truth['sigma_bg'],len(bl)); bg -= bg.mean()
    eta = beta[ci]+gamma[ci]*out.is_lock.to_numpy()+kappa[ci]*out.side.to_numpy()+harm[hi]+bg[bi]
    out['hits'] = rng.binomial(out.n_trials.to_numpy(int),expit(eta))
    truth['gamma_design'] = float(gamma.mean())
    rows = [dict(parameter=name,variable=name,coordinate=None,dimension=None,truth=value) for name,value in truth.items()]
    for name,values,dimension,levels in (('beta',beta,'config',cl),('gamma_config',gamma,'config',cl),
                                        ('kappa_config',kappa,'config',cl),('u_harm',harm,'harmonic',hl),
                                        ('u_bg',bg,'background',bl)):
        rows += [dict(parameter=f'{name}[{level}]',variable=name,coordinate=level,dimension=dimension,truth=float(value))
                 for level,value in zip(levels,values)]
    return out, pd.DataFrame(rows)


## 3. Preflight
Inspect reusable data and the number of additional forecasts before continuing. A matching A2 pilot is required for FULL. Synthetic tones here check the instrument only.


In [ ]:
if len(MODELS) != 15:
    raise ValueError("This revision preserves the 15-geometry A2 model; it is not the single-Amazon experiment")
design = frequency_design()
CHECKPOINT_IDENTITIES = {pl.model_tag(P,S):ml.checkpoint_identity(P,S) for P,S in MODELS}
paths = [BAYES_DIR/name for name in ("probe_lib.py","model_loader.py","checkpointing.py")]
paths += sorted((REPO/"chronos/data/synthetic").rglob("*.py"))
MEASUREMENT_HELPER_HASHES = {path.relative_to(REPO).as_posix():cp.sha256_file(path) for path in paths}
SOURCE = resolve_source()
test_f = np.array([32.,64.,96.])
assert hit_values(spectral_peaks(np.stack([pl.make_tone(f,.37,pl.PRED) for f in test_f])),test_f).all()
assert np.isnan(spectral_peaks(np.zeros((1,pl.PRED)))).all()
assert not hit_values(np.full((1,TOP_K),np.nan),[32.]).any()
functions = [value for name,value in list(globals().items()) if inspect.isfunction(value) and value.__module__=="__main__" and name!="display"]
SCIENCE_SPEC = dict(notebook=NOTEBOOK_VERSION,model=MODEL_VERSION,models=MODELS,generators=GENERATORS,
                    snr=TONE_SNR,top_k=TOP_K,tolerance_hz=TOL_HZ,nfft=NFFT,n_phase=N_PHASE,seed=SEED,scope="all_arms",
                    prior_scale=PRIOR_SCALE,baseline_scale=BASELINE_SCALE,nu=NU,prior_scales=PRIOR_SCALES,
                    hierarchy="original A2 priors; exact reference-frequency coordinates",
                    reference_hz=REFERENCE_HZ,parameterization="reference64-v1",
                    fine_ppc_min_coverage=FINE_PPC_MIN_COVERAGE,stress_recovery="sparse-v1",
                    estimand="equal-geometry mean log odds vs midpoint of lo/hi log odds",side_coding={"lo":-1,"lock":0,"hi":1},
                    likelihood_encoding="exact_role_separated_binomial_counts",support_log_or=SUPPORT_LOG_OR,
                    rope_log_or=ROPE_LOG_OR,opposite_log_or=-SUPPORT_LOG_OR,cutoff=PROB_CUTOFF,
                    rhat_max=RHAT_MAX,ess_pilot=PILOT_ESS_MIN,ess_full=FULL_ESS_MIN,
                    ppc_min_coverage=PPC_MIN_COVERAGE,sensitivity_max_spread=SENSITIVITY_MAX_SPREAD,
                    recovery_min_coverage=RECOVERY_MIN_COVERAGE,
                    design_hash=cp.fingerprint(design.to_dict("records")),helper_hashes=MEASUREMENT_HELPER_HASHES,
                    checkpoints=CHECKPOINT_IDENTITIES,executing_logic_sha256=logic_fingerprint(functions))
CORE_FINGERPRINT = cp.fingerprint(SCIENCE_SPEC)
if IS_FULL:
    if not PILOT_RESULT_PATH.is_file(): raise FileNotFoundError(f"Run A2 PILOT first: {PILOT_RESULT_PATH}")
    pilot = json.loads(PILOT_RESULT_PATH.read_text(encoding="utf-8"))
    if pilot.get("core_fingerprint")!=CORE_FINGERPRINT or pilot.get("pilot_route_ok") is not True:
        raise ValueError("FULL requires a matching A2 PILOT PASS")
ANALYSIS_SPEC = dict(science=SCIENCE_SPEC,run_id=RUN_ID,run_mode=RUN_MODE,n_bg=N_BG,source=SOURCE,
                     draws=DRAWS,tune=TUNE,chains=CHAINS,target_accept=TARGET_ACCEPT,
                     recovery_draws=RECOVERY_DRAWS,recovery_tune=RECOVERY_TUNE,ppc_draws=PPC_DRAWS,effect_draws=EFFECT_DRAWS,
                     backend=NUTS_BACKEND,nuts_init=NUTS_INIT,chain_storage="independent-v1",python=platform.python_version(),
                     packages={name:package_version(name) for name in ("numpy","scipy","pandas","pymc","arviz","nutpie","pytensor","torch","chronos-forecasting")})
ANALYSIS_FINGERPRINT = cp.fingerprint(ANALYSIS_SPEC)
if MANIFEST_PATH.is_file():
    previous = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    if previous.get("analysis_fingerprint")!=ANALYSIS_FINGERPRINT:
        raise ValueError("Code/settings/source changed. Choose a new RUN_ID; existing runs are never auto-healed.")
else:
    cp.atomic_json(MANIFEST_PATH,dict(analysis_fingerprint=ANALYSIS_FINGERPRINT,analysis_spec=ANALYSIS_SPEC,artifacts={}))
total = len(design)*len(GENERATORS)*N_BG
reusable = len(design)*len(GENERATORS)*(SOURCE["n_bg"] if SOURCE else 0)
baseline_total = len(MODELS)*len(GENERATORS)*N_BG
source_has_calibration = SOURCE is not None and "data/background_forecasts.parquet" in source_manifest(SOURCE).get("artifacts",{})
baseline_reusable = len(MODELS)*len(GENERATORS)*SOURCE["n_bg"] if source_has_calibration else 0
print("Source:",SOURCE["root"] if SOURCE else "fresh collection")
print(f"A2 trials: {total:,}; reusable tone forecasts: {reusable:,}; additional tone forecasts: {total-reusable:,}")
print(f"Background-only Chronos forecasts: {baseline_total:,}; reusable: {baseline_reusable:,}; additional: {baseline_total-baseline_reusable:,}")
print("Observed probability/count arrays are not stored once per posterior draw.")
print("Output:",OUTPUT_ROOT)


## 4. Collect A2 arms and calibration
Calibration rates are descriptive on the registered trial weights. A frequent background-only hit can indicate spontaneous forecast peaks at the target. It is not evidence that an injected tone was preserved. Background-truth and background-forecast rates are kept distinct.


In [ ]:
arms,background_forecasts = collect_A2(design)
validate_arms(arms,design)
groups = aggregate_trials(arms)
save_table(DATA_ROOT/"A_counts.parquet",groups)
if arms.h.nunique()<2: raise ValueError("All tone forecasts have the same hit outcome; inspect the instrument before fitting")
rates = calibration_summary(arms,["generator","role"])
geometry_rates = calibration_summary(arms,["model","role"])
save_table(DATA_ROOT/"calibration_by_generator.parquet",rates)
save_table(DATA_ROOT/"calibration_by_geometry.parquet",geometry_rates)
MEASUREMENT_OK = bool((rates.truth_minus_background_truth>0).all())
CALIBRATION_COMPLETE = len(background_forecasts)==len(MODELS)*len(GENERATORS)*N_BG
print(f"{len(arms):,} trials -> {len(groups):,} role-separated count groups")
display(rates)
display(geometry_rates)
print("Instrument separation:",MEASUREMENT_OK,"| paired background forecasts complete:",CALIBRATION_COMPLETE)


### 4.1 Exact likelihood check
At identical parameters, the count and Bernoulli joint log probabilities differ only by the Binomial combinatorial constant. Unequal phase/trial counts and distinct control roles are retained.


In [ ]:
small_arms = arms[arms.model.isin(arms.model.unique()[:3]) & (arms.bg_id<2)].copy()
sort_keys = ["model","P","S","overlap","generator","bg_id","f_lock","role","is_lock"]
small_arms = small_arms.sort_values(sort_keys).reset_index(drop=True)
small_counts = aggregate_trials(small_arms)
bern_model,count_model = model_A2(small_arms,encoding="bernoulli"),model_A2(small_counts)
bern_logp,count_logp = bern_model.compile_logp(),count_model.compile_logp()
point = count_model.initial_point()
constant = np.sum(gammaln(small_counts.n_trials+1)-gammaln(small_counts.hits+1)-gammaln(small_counts.n_trials-small_counts.hits+1))
for shift in (-.6,0.,.4):
    candidate = {key:np.array(value,copy=True) for key,value in point.items()}
    candidate["gamma_config"] = np.linspace(-.8,.6,len(candidate["gamma_config"]))+shift
    candidate["kappa_config"] = np.linspace(-.3,.2,len(candidate["kappa_config"]))
    np.testing.assert_allclose(count_logp(candidate)-bern_logp(candidate),constant,atol=1e-7,rtol=1e-9)
print("A2 Bernoulli/Binomial equivalence: PASS")
del small_arms,small_counts,bern_model,count_model,bern_logp,count_logp
gc.collect()


## 5. Synthetic parameter recovery

Two known-truth scenarios: the original moderate-effects recovery and a sparse scenario with baseline -8, frequency scale 5, positive lock effects, and up to 10 backgrounds per generator. The latter checks the frequency/background effects and scales as well as geometry contrasts. Both require convergence, at least 80% interval coverage and coverage of `gamma_design`. These are synthetic checks, not evidence about Chronos.


In [ ]:
recovery_frame,recovery_truth = recovery_data(groups)
recovery_idata = fit_or_load("synthetic_parameter_recovery_A2",recovery_frame,draws=RECOVERY_DRAWS,tune=RECOVERY_TUNE)
ess_threshold = FULL_ESS_MIN if IS_FULL else PILOT_ESS_MIN
recovery_diagnostic,recovery_diagnostics = diagnostics_for(recovery_idata,ess_threshold)
recovery_rows = []
for row in recovery_truth.itertuples(index=False):
    array = recovery_idata.posterior[row.variable]
    if row.coordinate is not None and not pd.isna(row.coordinate): array = array.sel({row.dimension if hasattr(row,"dimension") else "config":row.coordinate})
    low,median,high = np.quantile(np.asarray(array).ravel(),[.025,.5,.975])
    recovery_rows.append(dict(parameter=row.parameter,truth=row.truth,median=float(median),eti_low=float(low),eti_high=float(high),covered=bool(low<=row.truth<=high)))
recovery_table = pd.DataFrame(recovery_rows)
headline_covered = bool(recovery_table.set_index("parameter").loc["gamma_design","covered"])
RECOVERY_OK = bool(recovery_diagnostic["passed"] and headline_covered and recovery_table.covered.mean()>=RECOVERY_MIN_COVERAGE)
save_table(OUTPUT_ROOT/"synthetic_recovery_summary.parquet",recovery_table)
save_table(OUTPUT_ROOT/"synthetic_recovery_diagnostics.parquet",recovery_diagnostics.reset_index(names="parameter"))
display(recovery_table)
print("SYNTHETIC ONLY:",recovery_diagnostic,"coverage:",recovery_table.covered.mean(),"gate:",RECOVERY_OK)
del recovery_idata,recovery_frame
gc.collect()


stress_frame, stress_truth = sparse_recovery_data(groups)
stress_idata = fit_or_load('synthetic_sparse_recovery_A2',stress_frame,draws=RECOVERY_DRAWS,tune=RECOVERY_TUNE)
stress_diagnostic, stress_diagnostics = diagnostics_for(stress_idata,ess_threshold)
stress_rows = []
for row in stress_truth.itertuples(index=False):
    array = stress_idata.posterior[row.variable]
    if row.coordinate is not None and not pd.isna(row.coordinate):
        array = array.sel({row.dimension:row.coordinate})
    low,median,high = np.quantile(np.asarray(array).ravel(),[.025,.5,.975])
    stress_rows.append(dict(parameter=row.parameter,truth=row.truth,median=float(median),eti_low=float(low),
                            eti_high=float(high),covered=bool(low<=row.truth<=high)))
stress_table = pd.DataFrame(stress_rows)
STRESS_RECOVERY_OK = bool(stress_diagnostic['passed'] and stress_table.set_index('parameter').loc['gamma_design','covered']
                          and stress_table.covered.mean()>=RECOVERY_MIN_COVERAGE)
save_table(OUTPUT_ROOT/'synthetic_sparse_recovery_summary.parquet',stress_table)
save_table(OUTPUT_ROOT/'synthetic_sparse_recovery_diagnostics.parquet',stress_diagnostics.reset_index(names='parameter'))
print('SPARSE SYNTHETIC ONLY:',stress_diagnostic,'coverage:',stress_table.covered.mean(),'gate:',STRESS_RECOVERY_OK)
print('Synthetic hit rate:',stress_frame.hits.sum()/stress_frame.n_trials.sum(),
      '| all-zero frequencies:',int(stress_frame.groupby('f_lock').hits.sum().eq(0).sum()))
display(stress_table)
del stress_idata, stress_frame
gc.collect()


## 6. Primary A2 fit

Full posterior inference uses all selected data. Every chain is checkpointed separately. Trace lines print every 200 iterations without relying on ipywidgets. `gamma_design` remains the equal-weight mean over the 15 geometry effects. All gates, including the additional fine PPC and sparse recovery, must pass before final reporting.


In [ ]:
idata_A2 = fit_or_load("primary_A2_logit",groups)
primary_diagnostic,parameter_diagnostics = diagnostics_for(idata_A2,ess_threshold)
PRIMARY_OK = primary_diagnostic["passed"]
primary_effect = effect_summary(posterior_array(idata_A2,"gamma_design"))
effect_table = geometry_effect_table(idata_A2,groups)
save_table(OUTPUT_ROOT/"primary_diagnostics.parquet",parameter_diagnostics.reset_index(names="parameter"))
save_table(OUTPUT_ROOT/"geometry_effects.parquet",effect_table)
save_table(OUTPUT_ROOT/"design_effect.parquet",pd.DataFrame([primary_effect]))
display(pd.DataFrame([primary_effect]))
display(effect_table)
print("Primary convergence:",primary_diagnostic)
display(parameter_diagnostics.sort_values("ess_bulk").head(12))


### 6.1 Traces and geometry effects
Intervals are 95% equal-tail posterior intervals. The vertical zero line corresponds to equal lock odds and geometric-mean control odds; it is not a claim about pooled control probabilities.


In [ ]:
trace_names = ["gamma_design","gamma_bar","sigma_gamma","kappa_bar","sigma_kappa","beta_bar","tau","sigma_harm","sigma_bg"]
fig,axes = plt.subplots(len(trace_names),1,figsize=(11,15),sharex=True)
for axis,name in zip(axes,trace_names):
    for chain,values in enumerate(np.asarray(idata_A2.posterior[name].transpose("chain","draw"))): axis.plot(values,lw=.55,label=f"chain {chain+1}")
    axis.set_ylabel(name)
axes[0].legend(ncol=CHAINS,fontsize=8)
axes[-1].set_xlabel("retained draw")
fig.suptitle(f"A2, {RUN_MODE}"); fig.tight_layout()
fig.savefig(FIGURE_ROOT/"primary_traces.png",dpi=130); plt.show(); plt.close(fig)
ordered = effect_table.sort_values("median_log_or").reset_index(drop=True)
fig,axis = plt.subplots(figsize=(9,7))
axis.hlines(np.arange(len(ordered)),ordered.eti_low_log_or,ordered.eti_high_log_or,color="tab:blue")
axis.scatter(ordered.median_log_or,np.arange(len(ordered)),s=28,color="tab:blue")
axis.axvline(0,color="black",lw=1); axis.axvline(SUPPORT_LOG_OR,color="tab:red",ls="--",lw=.8)
axis.set_yticks(np.arange(len(ordered)),ordered.model)
axis.set_xlabel("Lock log odds minus midpoint of control log odds")
axis.set_title(f"A2 geometry effects, {RUN_MODE}")
fig.tight_layout(); fig.savefig(FIGURE_ROOT/"geometry_effects.png",dpi=140); plt.show(); plt.close(fig)


### 6.2 PPC: marginal checks and geometry x frequency x role

The original 34 and 51 checks are retained. A third set checks each observed geometry-frequency-role cell (615 cells in the full design); it also requires at least 90% of observed rates to lie in their 95% predictive intervals. This is an additional aggregate model-adequacy check, not a trial-level accuracy or a multiple-testing claim.


In [ ]:
ppc_table = predictive_check(idata_A2,groups)
save_table(OUTPUT_ROOT/"primary_ppc.parquet",ppc_table)
ppc_summary = ppc_table.groupby("check_set").passed.agg(["sum","size","mean"])
PPC_OK = bool(ppc_summary.index.tolist()==["A1_comparable","separate_roles"] and (ppc_summary["mean"]>=PPC_MIN_COVERAGE).all())
display(ppc_summary)
display(ppc_table[~ppc_table.passed])
comparison = comparison_to_A1(ppc_table)
if not comparison.empty:
    save_table(OUTPUT_ROOT/"A1_A2_same_data_ppc.parquet",comparison)
    print("Same-data original checks: A1",comparison.passed_A1.mean(),"A2",comparison.passed_A2.mean())
    display(comparison[["key","passed_A1","passed_A2","observed_rate_A2","rep_low_A2","rep_high_A2"]])
else:
    print("No A1 PPC checkpoint for exactly this data population; no cross-run percentage comparison issued.")
print("A2 PPC gate:",PPC_OK)


fine_ppc = fine_predictive_check(idata_A2, groups)
save_table(OUTPUT_ROOT/"primary_fine_ppc.parquet", fine_ppc)
FINE_PPC_OK = bool(fine_ppc.passed.mean() >= FINE_PPC_MIN_COVERAGE)
print("Geometry x frequency x role PPC:",int(fine_ppc.passed.sum()),"/",len(fine_ppc),"gate:",FINE_PPC_OK)
display(fine_ppc[~fine_ppc.passed])


## 7. FULL prior and link sensitivity
The primary prior scale is 0.5. FULL adds 0.25, 1.0 and probit with approximately matched prior widths. Probit is converted to the same within-geometry control-log-odds midpoint contrast, then averaged equally over geometries. Up to 2,000 posterior draws are used for this nonlinear conversion. All variants must converge and support/equivalence/opposite probabilities must vary by at most 0.10.


In [ ]:
sensitivity_rows = []
if IS_FULL and PRIMARY_OK:
    sensitivity_rows.append(dict(variant="primary logit .5",passed=PRIMARY_OK,**primary_effect))
    for label,scale,link in (("logit .25",.25,"logit"),("logit 1.0",1.,"logit"),("probit matched .5",.5,"probit")):
        fit = fit_or_load("sensitivity_"+label.replace(" ","_"),groups,scale=scale,link=link)
        diagnostic,_ = diagnostics_for(fit,FULL_ESS_MIN)
        converted = comparable_effects(fit,groups,link=link)
        effect = effect_summary(converted.mean(axis=1))
        sensitivity_rows.append(dict(variant=label,passed=diagnostic["passed"],**effect))
        del fit; gc.collect()
    sensitivity_table = pd.DataFrame(sensitivity_rows)
    columns = ["p_support","p_equivalence","p_opposite"]
    spread = sensitivity_table[columns].max()-sensitivity_table[columns].min()
    SENSITIVITY_OK = bool(sensitivity_table.passed.all() and (spread<=SENSITIVITY_MAX_SPREAD).all())
    save_table(OUTPUT_ROOT/"sensitivity.parquet",sensitivity_table)
    display(sensitivity_table); print("Probability spreads:",spread.to_dict())
else:
    sensitivity_table = pd.DataFrame(); SENSITIVITY_OK = False
    print("Sensitivity deferred: PILOT or failed primary convergence.")


## 8. Final result
PILOT is non-reportable. FULL requires instrument separation, complete paired background forecasts, synthetic recovery, convergence, both PPC sets and sensitivity. An opposite-direction result is allowed. Background-only forecasts inform interpretation; no favorable calibration outcome is required or silently selected.


In [ ]:
PILOT_ROUTE_OK = bool(MEASUREMENT_OK and CALIBRATION_COMPLETE and RECOVERY_OK and PRIMARY_OK and PPC_OK and FINE_PPC_OK and STRESS_RECOVERY_OK)
FINAL_GATE_OK = bool(IS_FULL and PILOT_ROUTE_OK and SENSITIVITY_OK)
gate_table = pd.DataFrame([dict(gate=name,passed=bool(value)) for name,value in [
    ("instrument separation",MEASUREMENT_OK),("paired background forecasts",CALIBRATION_COMPLETE),
    ("synthetic A2 recovery",RECOVERY_OK),("A2 convergence",PRIMARY_OK),("both A2 PPC sets",PPC_OK),
    ("fine geometry-frequency-role PPC",FINE_PPC_OK),("sparse synthetic recovery",STRESS_RECOVERY_OK),
    ("FULL mode",IS_FULL),("prior/link sensitivity",SENSITIVITY_OK)]])
result = dict(run_mode=RUN_MODE,model_version=MODEL_VERSION,core_fingerprint=CORE_FINGERPRINT,
              analysis_fingerprint=ANALYSIS_FINGERPRINT,pilot_route_ok=PILOT_ROUTE_OK,gate_ok=FINAL_GATE_OK,
              verdict=verdict_from(FINAL_GATE_OK,primary_effect),scope="all_arms",
              estimand="equal-geometry mean log odds versus midpoint of lo/hi log odds",effect=primary_effect,
              primary_diagnostic=primary_diagnostic,gates=gate_table.to_dict("records"))
cp.atomic_json(OUTPUT_ROOT/"final_verdict.json",result); record_artifact(OUTPUT_ROOT/"final_verdict.json")
save_table(OUTPUT_ROOT/"gates.parquet",gate_table)
display(gate_table); print("FINAL A2:",result["verdict"])
if not IS_FULL:
    print("A2 PILOT PASS: change RUN_MODE to FULL in a clean runtime." if PILOT_ROUTE_OK else "A2 PILOT FAIL: inspect the failed gates before FULL.")
print("Artifacts:",OUTPUT_ROOT)


The saved notebook is unexecuted. Synthetic recovery, actual-data benchmarks, and empirical posterior inference are distinct. FULL reporting requires the matching revised PILOT, complete posterior samples, both recoveries, all three PPC sets, and converged prior/link sensitivities. The original localisation measurement and effect thresholds are unchanged.